# Phase 2 labeling — Colab GPU runbook (free Qwen frontier labeler)

Reads the **frozen corpus** from Drive and runs `labeling/label.py` with the
free **Qwen** frontier labeler loaded via transformers (4-bit) on the **T4**,
then writes `labeled_corpus.jsonl` (+ a spot-check sample) back to Drive.

This is still the LOCAL pipeline half — it reads only the frozen file and never
re-invokes the swarm. Run the generator notebook first.

**Runtime → Change runtime type → GPU (T4)** before running.

In [ ]:
!nvidia-smi -L
!pip -q install "transformers>=4.44" accelerate bitsandbytes pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
PROJECT = '/content/swarm_alignment'   # clone/upload the repo here
assert os.path.isdir(PROJECT), f'upload/clone the repo to {PROJECT} first'
sys.path.insert(0, PROJECT); os.chdir(PROJECT)

In [ ]:
# Point paths at Drive and select the free Qwen labeler on the GPU.
from common.util import load_config
cfg = load_config('configs/default.yaml')

drive_root = '/content/drive/MyDrive/swarm_alignment'
cfg['paths']['frozen_corpus'] = os.path.join(drive_root, 'frozen_corpus.jsonl')
cfg['paths']['labeled_corpus'] = os.path.join(drive_root, 'labeled_corpus.jsonl')

cfg['labeling']['frontier_labeler'] = 'qwen'
cfg['labeling']['qwen']['backend'] = 'transformers'          # load on the GPU
cfg['labeling']['qwen']['model'] = 'Qwen/Qwen2.5-3B-Instruct'  # HF repo id
cfg['labeling']['qwen']['quantization'] = '4bit'
# Tip: set max_calls to a small number first to sanity-check quality fast.
cfg['labeling']['qwen']['max_calls'] = 30
print('labeler:', cfg['labeling']['frontier_labeler'], '| model:', cfg['labeling']['qwen']['model'])

In [ ]:
# Quick sanity pass (max_calls cap). Inspect the spot-check dump before the full run.
from labeling.label import label_corpus
res = label_corpus(cfg)
res['counts']

In [ ]:
# Eyeball the spot-check sample (low-confidence / high-tier rows).
import json, glob
sc = cfg['paths']['labeled_corpus'].replace('.jsonl', '_spotcheck.json')
print(json.dumps(json.load(open(sc))[:10], indent=2, ensure_ascii=False))

In [ ]:
# Happy with quality? Remove the cap and run the full labeling pass.
cfg['labeling']['qwen']['max_calls'] = 0   # unlimited
res = label_corpus(cfg)
print(res['counts'])
print('labeled corpus ->', cfg['paths']['labeled_corpus'])
# Download labeled_corpus.jsonl to your laptop's artifacts/ to continue the
# local pipeline (adaption.run -> eval), or keep going on Colab.